# Quantum Billiards — rectangle and stadium

Solve $-\nabla^2\psi = E\psi$ with $\psi = 0$ on the wall, by finite differences, with
the boundary condition imposed **by masking**.

The shape enters only through a boolean array. `rectangle(...)` and `stadium(...)`
each build a grid and a mask; everything downstream — the Laplacian, the eigensolver,
the plots — is identical for both and never asks which shape it has.

The grid is a **padded box strictly larger than the shape**. The padding nodes are
outside the mask, so they are not unknowns; the shape floats in a sea of wall. This is
what lets one solver handle any geometry.

The rectangle has an exact answer,
$E_{nm} = \pi^2(n^2/L_x^2 + m^2/L_y^2)$, so it is the calibration. The stadium has no
closed form, so it is checked against Weyl's law instead.

Units are natural, $\hbar = m = 1$, and we solve $-\nabla^2\psi = E\psi$ exactly as
posed (no factor of $1/2$), so $E$ has units of (length)$^{-2}$.

In [ ]:
from dataclasses import dataclass, field

import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
import scipy.sparse.linalg as spla

PHI = (1 + np.sqrt(5)) / 2

h = 0.02            # <-- THE KNOB. coarse: 0.05   default: 0.02   fine: 0.01
n_states = 40       # how many eigenvalues to ask for
PAD = 4             # cells of empty space between the shape and the grid edge

## The two shapes

Both return the same `Billiard` record, so they are interchangeable everywhere below.

**Rectangle.** Built from index ranges rather than a coordinate test, so the wall ring
lands *exactly* on nodes with no floating-point ambiguity — that is what keeps the
exact-spectrum comparison clean. The side lengths are snapped to a whole number of
cells, so the rectangle actually solved may differ from the one requested by less than
one $h$; `b.params` reports what was really used, and the exact formula uses those.

**Stadium.** Two parallel straight walls of length $a$ closed by two semicircles of
radius $r$ — total width $a + 2r$, total height $2r$. The inside test is the one-liner

$$\big(\max(|u| - a/2,\, 0)\big)^2 + v^2 < r^2,$$

which is just "distance from the central segment of length $a$ is less than $r$". It
gives the flat walls and the semicircular caps in a single expression, with no case
split to get wrong at the join.

> **Why the stadium's coordinates are built from integer offsets.** A stadium has two
> mirror symmetries, so every eigenstate must be even or odd about each axis. Writing
> the inside test with `X - cx` destroys that: the flat walls sit at $|v| = r$ exactly,
> and in floating point `0.08 - 0.58 = -0.49999999999999994` while
> `1.08 - 0.58 = 0.5000000000000001`, so the bottom wall tested *inside* and the top
> wall *outside*. The mask came out one cell taller along the bottom and every
> eigenstate was visibly lopsided. Building $u$ as `(i - ic) * h` makes
> $u_{-k} = -u_{k}$ bit for bit, and the two `assert`s in `stadium()` now hold
> exactly.

In [ ]:
@dataclass
class Billiard:
    """A grid, a mask, and the exact geometry the mask is approximating."""
    name: str
    h: float
    x: np.ndarray            # node coordinates, 1D
    y: np.ndarray
    mask: np.ndarray         # (len(x), len(y)) bool; True = unknown, False = wall
    area: float              # continuum area of the true shape
    perim: float             # continuum perimeter
    outline: tuple           # (xs, ys) of the true boundary, for plotting
    params: dict = field(default_factory=dict)

    @property
    def XY(self):
        return np.meshgrid(self.x, self.y, indexing="ij")

    @property
    def n_unknown(self):
        return int(self.mask.sum())


def _grid(ncx, ncy, h):
    """Node grid with ncx by ncy cells, spacing h. Coordinates are exact multiples of h."""
    x, y = np.arange(ncx + 1) * h, np.arange(ncy + 1) * h
    return x, y


def rectangle(Lx=1.0, Ly=1.0 / PHI, h=h, pad=PAD):
    """Lx by Ly rectangle, snapped to the grid, floating in a padded box."""
    nx, ny = round(Lx / h), round(Ly / h)          # shape size in cells
    Lx_eff, Ly_eff = nx * h, ny * h
    x, y = _grid(nx + 2 * pad, ny + 2 * pad, h)

    mask = np.zeros((x.size, y.size), dtype=bool)
    mask[pad + 1:pad + nx, pad + 1:pad + ny] = True   # strictly interior nodes

    x0, y0 = x[pad], y[pad]
    corners = ([x0, x0 + Lx_eff, x0 + Lx_eff, x0, x0],
               [y0, y0, y0 + Ly_eff, y0 + Ly_eff, y0])
    return Billiard("rectangle", h, x, y, mask,
                    area=Lx_eff * Ly_eff, perim=2 * (Lx_eff + Ly_eff),
                    outline=corners, params=dict(Lx=Lx_eff, Ly=Ly_eff))


def stadium(r=0.5, a=1.0, h=h, pad=PAD):
    """Bunimovich stadium: straight walls of length a closed by semicircles of radius r.

    Width a + 2r, height 2r. Default a/r = 2.
    """
    # An EVEN number of cells each way, so the centre of symmetry lands on a node.
    ncx, ncy = 2 * int(np.ceil((a + 2 * r) / (2 * h))), 2 * int(np.ceil(r / h))
    x, y = _grid(ncx + 2 * pad, ncy + 2 * pad, h)
    ic, jc = pad + ncx // 2, pad + ncy // 2            # index of the centre node
    cx, cy = x[ic], y[jc]

    # Coordinates RELATIVE to the centre, as exact integer multiples of h. Computing
    # them as (i - ic)*h rather than as x[i] - cx makes u[ic-k] = -u[ic+k] bit for bit,
    # so the mask comes out exactly mirror-symmetric. Subtracting two floats does not:
    # at h = 0.02 that put one flat wall inside the mask and the other outside, which
    # made the stadium one cell taller along the bottom and broke every eigenstate's
    # up-down symmetry.
    U, V = np.meshgrid((np.arange(x.size) - ic) * h,
                       (np.arange(y.size) - jc) * h, indexing="ij")
    d = np.maximum(np.abs(U) - a / 2, 0.0)             # horizontal distance to the segment
    mask = d**2 + V**2 < r**2                          # distance to segment < r

    assert (mask == mask[:, ::-1]).all(), "stadium mask not symmetric about the x axis"
    assert (mask == mask[::-1, :]).all(), "stadium mask not symmetric about the y axis"

    t = np.linspace(-np.pi / 2, np.pi / 2, 200)
    xs = np.concatenate([cx + a / 2 + r * np.cos(t), cx - a / 2 - r * np.cos(t)])
    ys = np.concatenate([cy + r * np.sin(t), cy - r * np.sin(t)])
    outline = (np.append(xs, xs[0]), np.append(ys, ys[0]))

    return Billiard("stadium", h, x, y, mask,
                    area=2 * r * a + np.pi * r**2, perim=2 * a + 2 * np.pi * r,
                    outline=outline, params=dict(r=r, a=a))


for b in (rectangle(), stadium()):
    filled = b.n_unknown * b.h**2
    print(f"{b.name:10s} grid {b.mask.shape[0]}x{b.mask.shape[1]}  "
          f"unknowns {b.n_unknown:6d}  area {b.area:.4f} (mask {filled:.4f})  {b.params}")

### Look at the borders

The mask *is* the physics, and it is the easiest thing to get silently wrong, so plot
it before trusting anything. The red curve is the true boundary; the blue cells are the
unknowns. The rectangle lies exactly on nodes. The stadium's caps **staircase**, and
that staircase is a real perturbation to the spectrum, not a cosmetic issue — it is why
a curved wall converges more slowly than a straight one.

In [ ]:
def plot_mask(b, ax, grid_lines=True):
    kw = dict(edgecolors="0.8", linewidth=0.15) if grid_lines else {}
    ax.pcolormesh(b.x, b.y, b.mask.T.astype(float), cmap="Blues", vmin=0, vmax=1.6, **kw)
    ax.plot(*b.outline, "r-", lw=1.8)
    ax.set_aspect("equal")
    ax.set_title(f"{b.name}: {b.n_unknown} unknowns", fontsize=10)


fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_mask(rectangle(), axes[0])
plot_mask(stadium(), axes[1])
fig.suptitle("masks, with the true boundary in red (note the padding ring of wall)")
plt.tight_layout()
plt.show()

## The generic solver

Standard 5-point stencil for $-\nabla^2$: centre point plus both neighbours in each
direction,

$$(-\nabla^2\psi)_{ij} \approx \frac{4\psi_{ij} - \psi_{i+1,j} - \psi_{i-1,j} - \psi_{i,j+1} - \psi_{i,j-1}}{h^2}.$$

Dirichlet is built in by **omission**: a neighbour outside the mask has $\psi = 0$, so
its term is simply not added. Nothing here knows what shape it is solving.

In [ ]:
def build_laplacian(mask, h):
    """-grad^2 restricted to the True cells of `mask`. Returns (sparse matrix, index map).

    idx[i,j] is the row/column of node (i,j), or -1 if that node is not an unknown.
    """
    assert not mask[0].any() and not mask[-1].any() and \
           not mask[:, 0].any() and not mask[:, -1].any(), \
        "mask touches the grid edge -- increase pad, or the array edge becomes a wall"

    idx = -np.ones(mask.shape, dtype=int)
    idx[mask] = np.arange(mask.sum())

    rows, cols, vals = [], [], []
    for i, j in zip(*np.nonzero(mask)):
        k = idx[i, j]
        rows.append(k); cols.append(k); vals.append(4 / h**2)
        for ii, jj in ((i + 1, j), (i - 1, j), (i, j + 1), (i, j - 1)):
            if mask[ii, jj]:                      # padding guarantees this is in range
                rows.append(k); cols.append(idx[ii, jj]); vals.append(-1 / h**2)

    n = mask.sum()
    return sp.csc_matrix((vals, (rows, cols)), shape=(n, n)), idx


def solve(b, k):
    """Lowest k eigenpairs of any Billiard. psi normalised so sum |psi|^2 h^2 = 1."""
    A, _ = build_laplacian(b.mask, b.h)
    E, V = spla.eigsh(A, k=k, sigma=0.0, which="LM")
    order = np.argsort(E)
    return E[order], V[:, order] / b.h


def to_grid(vec, mask):
    """Scatter a solution vector back onto the full node grid (zeros on the wall)."""
    out = np.zeros(mask.shape)
    out[mask] = vec
    return out


rect = rectangle()
E_rect, V_rect = solve(rect, n_states)
print(f"rectangle: {n_states} states on {rect.n_unknown} unknowns")

## Check 1 — the rectangle against its exact spectrum

The right-hand column comes from analysis, not from the code, so this would fail
loudly for a wrong stencil, a wrong $h$, or a mask off by one cell.

In [ ]:
def exact_levels(Lx, Ly, n_max=60):
    """Sorted list of (E, n, m) for the rectangle."""
    n, m = np.meshgrid(np.arange(1, n_max + 1), np.arange(1, n_max + 1), indexing="ij")
    E = np.pi**2 * (n**2 / Lx**2 + m**2 / Ly**2)
    return sorted(zip(E.ravel(), n.ravel(), m.ravel()))


levels = exact_levels(**rect.params)[:n_states]
E_exact = np.array([e for e, _, _ in levels])
rel_err = (E_rect - E_exact) / E_exact

print("  k  (n,m)      E_exact        E_numeric      rel. error")
print("  " + "-" * 55)
for k, ((ex, n, m), en, re) in enumerate(zip(levels, E_rect, rel_err)):
    print(f"{k:3d}  ({n},{m})   {ex:12.5f}   {en:12.5f}   {re:+10.2e}")

In [ ]:
# (a) The sign is not an accident. The 5-point Dirichlet Laplacian has eigenvalues
#     (4/h^2) sin^2(k h / 2) < k^2, so EVERY level must come out BELOW the exact one,
#     at any resolution. A positive error is a bug, not a discretisation effect.
assert np.all(E_rect < E_exact), "a finite-difference level came out above the exact one"

# (b) The low states should be accurate at the default resolution.
assert np.abs(rel_err[:6]).max() < 0.02, f"low states off by {np.abs(rel_err[:6]).max():.1%}"

# (c) The snapped rectangle is nx by ny cells, so E ~ n^2 ny^2 + m^2 nx^2. With nx, ny
#     coprime a degeneracy needs n^2 - n'^2 divisible by nx^2, impossible in this range.
gaps = np.diff(E_exact)
assert gaps.min() > 1e-6 * E_exact[-1], "unexpected degeneracy -- check the aspect ratio"

print("checks passed")
print(f"worst error over {n_states} states: {np.abs(rel_err).max():.2%} "
      f"(state {np.argmax(np.abs(rel_err))})")
print(f"smallest exact gap / mean gap: {gaps.min() / gaps.mean():.4f}")

## Check 2 — convergence is second order

The errors above are discretisation, not mistakes, so they must shrink predictably.
The 5-point stencil is $O(h^2)$: halving $h$ should cut the error by **4**. A ratio of
2 would mean the stencil or the spacing is wrong even though the numbers looked fine.

Each resolution snaps to its own whole number of cells, so each is compared against the
exact spectrum of the rectangle it actually solved.

In [ ]:
n_conv = 6
print("    h      Lx_eff   Ly_eff   unknowns   mean |rel err| (lowest 6)   ratio")
print("    " + "-" * 72)
prev = None
for hc in (4 * h, 2 * h, h, h / 2):
    bc = rectangle(h=hc)
    Ec, _ = solve(bc, n_conv)
    ref = np.array([e for e, _, _ in exact_levels(**bc.params)[:n_conv]])
    err = np.abs((Ec - ref) / ref).mean()
    ratio = "" if prev is None else f"{prev / err:6.2f}"
    print(f"  {hc:7.4f}  {bc.params['Lx']:.4f}   {bc.params['Ly']:.4f}  "
          f"{bc.n_unknown:8d}        {err:.4e}          {ratio}")
    prev = err

## Check 3 — Weyl's law, which works for *any* shape

The stadium has no closed-form spectrum, so it needs a different check. The number of
levels below $E$ obeys

$$N(E) \;\approx\; \frac{A}{4\pi}E \;-\; \frac{P}{4\pi}\sqrt{E},$$

with $A$ the area and $P$ the perimeter — the leading term is phase-space volume, the
correction is the Dirichlet wall. This is non-trivial: it ties the *spectrum* to the
*geometry*, so it catches an area that does not match the mask, a wrong $h$, or missed
or duplicated states. The staircase should track the curve to within $O(1)$; it will
not sit on it, and the residual wiggle is itself physical.

In [ ]:
stad = stadium()
E_stad, V_stad = solve(stad, n_states)
print(f"stadium: {n_states} states on {stad.n_unknown} unknowns")


def weyl(E, b):
    return b.area * E / (4 * np.pi) - b.perim * np.sqrt(E) / (4 * np.pi)


fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (b, Eb) in zip(axes, ((rect, E_rect), (stad, E_stad))):
    counted = np.arange(1, len(Eb) + 1)
    ax.step(Eb, counted, where="post", label="counted $N(E)$")
    Eg = np.linspace(Eb[0], Eb[-1], 400)
    ax.plot(Eg, weyl(Eg, b), "r--", label="Weyl")
    ax.set_xlabel("$E$"); ax.set_ylabel("$N(E)$"); ax.set_title(b.name); ax.legend()

    dev = np.abs(counted - weyl(Eb, b)).max()
    print(f"{b.name:10s} A = {b.area:.4f}  P = {b.perim:.4f}   max |N - Weyl| = {dev:.2f}")
    assert dev < 5, f"{b.name}: spectrum and geometry disagree by {dev:.1f} levels"
plt.tight_layout()
plt.show()

## The states

Rectangle states are $\sin(n\pi x/L_x)\sin(m\pi y/L_y)$ — count the nodal lines and
they should match the $(n,m)$ in the title, which is a third, visual check on the
ordering. The stadium's nodal lines do not form a grid; that is the whole point.

In [ ]:
def plot_states(b, E, V, which, labels=None, ncol=3):
    X, Y = b.XY
    nrow = int(np.ceil(len(which) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 2.6 * nrow))
    for ax, k in zip(np.atleast_1d(axes).ravel(), which):
        psi = to_grid(V[:, k], b.mask)
        lim = np.abs(psi).max()
        ax.pcolormesh(b.x, b.y, psi.T, cmap="RdBu_r", vmin=-lim, vmax=lim, shading="auto")
        ax.contour(X, Y, psi, levels=[0.0], colors="k", linewidths=0.5)
        ax.plot(*b.outline, "k-", lw=0.8)
        tag = "" if labels is None else f"  {labels[k]}"
        ax.set_title(f"k={k}{tag}  E={E[k]:.1f}", fontsize=9)
        ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"{b.name} eigenstates (black = nodal lines)")
    plt.tight_layout()
    plt.show()


plot_states(rect, E_rect, V_rect, range(6),
            labels=[f"(n,m)=({n},{m})" for _, n, m in levels])
plot_states(stad, E_stad, V_stad, range(6))

In [ ]:
# Normalisation and orthogonality for both shapes -- cheap, but they pin down the cell
# area, which would otherwise silently corrupt the inverse participation ratio later.
for b, V in ((rect, V_rect), (stad, V_stad)):
    norms = (V**2).sum(axis=0) * b.h**2
    overlaps = (V.T @ V) * b.h**2 - np.eye(V.shape[1])
    print(f"{b.name:10s} max |<k|k> - 1| = {np.abs(norms - 1).max():.2e}   "
          f"max |<j|k>| = {np.abs(overlaps).max():.2e}")
    assert np.abs(norms - 1).max() < 1e-10
    assert np.abs(overlaps).max() < 1e-8

---

**Stop here.** Both shapes run the same solver, the stadium's localised states are
ranked and plotted, and five independent checks hold (exact rectangle spectrum,
$O(h^2)$ convergence, Weyl's law, orthonormality, exact rectangle IPR).

Still to come:

- A **quarter stadium**. The mask is now exactly mirror-symmetric in both axes, which
  is correct physics — and precisely why the full stadium's spectrum is four
  independent spectra laid on top of one another. Superposing uncorrelated spectra
  destroys level repulsion, so the level statistics of *this* table will come out
  looking integrable even though it is chaotic. Cutting the table in quarter with hard
  walls on the symmetry axes isolates one sector.
- **Level statistics**: unfold to unit mean spacing (Weyl's law, already implemented,
  is the unfolding function), histogram the nearest-neighbour gaps, and compare
  Poisson against Wigner for the two shapes.
- A **real periodic orbit** from root-finding the bounce map, laid over a scarred
  state.

In [ ]:
def ipr(V, b):
    """Inverse participation ratio, made dimensionless by multiplying by the area.

    I = sum_i |psi_i|^4 h^2  ~  1 / A_eff, so I*A is (roughly) the reciprocal of the
    FRACTION of the table the state occupies: spread out -> small, piled up -> large.

    Three benchmarks make the number readable:
        rectangle eigenstate   I*A = 9/4 = 2.25   exactly, for every (n,m)
        random GOE state       I*A -> 3           (Porter-Thomas, real amplitudes)
        scarred state          I*A > 3
    """
    return (V**4).sum(axis=0) * b.h**2 * b.area


I_rect = ipr(V_rect, rect)
I_stad = ipr(V_stad, stad)
print(f"rectangle  I*A: min {I_rect.min():.6f}  max {I_rect.max():.6f}   (exact 9/4)")
print(f"stadium    I*A: min {I_stad.min():.3f}  median {np.median(I_stad):.3f}  "
      f"max {I_stad.max():.3f}")

# The rectangle value is EXACT, not approximate: psi^4 ~ sin^4 = 3/8 - cos(2t)/2 +
# cos(4t)/8, and the cosine terms sum to exactly zero over a uniform grid until the
# mode aliases. So (2/sqrt(LxLy))^4 * (3Lx/8)(3Ly/8) = 9/(4 LxLy), i.e. I*A = 9/4 to
# machine precision for every low state. That pins the cell area inside the IPR --
# get h^2 wrong and every localisation number below is wrong by a constant factor.
assert np.abs(I_rect[:10] - 2.25).max() < 1e-9, "IPR normalisation is wrong"

# Eigenstates of a symmetric table must be even or odd about each mirror axis.
# This is what the asymmetric-mask bug used to break, so it is worth asserting.
for k in range(6):
    p = to_grid(V_stad[:, k], stad.mask)
    defect = min(np.abs(p - p[:, ::-1]).max(), np.abs(p + p[:, ::-1]).max())
    assert defect < 1e-12 * np.abs(p).max(), f"state {k} is neither even nor odd in y"

print("checks passed")

## The scar hunt

Scars are a **semiclassical** effect: they need the wavelength to be short compared to
the table, so the low states above are useless for this. This section re-solves the
stadium on a finer grid and asks for several hundred states, then sorts by $I\cdot A$.

What you should see at the top of the ranking:

- **Vertical stripe states** filling the straight section — the particle bouncing
  straight up and down between the two flat walls. These dominate. Strictly they are
  *not* Heller scars: the vertical bounce in a stadium is a **marginally stable**
  one-parameter family of orbits (neutral, not unstable), which is exactly why they
  localise so strongly. They are a real and famous feature of the stadium spectrum,
  and worth naming correctly rather than calling every bright state a scar.
- **Whispering-gallery states** hugging the curved walls.
- Genuine scars on *unstable* orbits — the bowtie, the horizontal bounce, the V — are
  fainter and sit further down the list.

And the honest caveat: **IPR is a blunt instrument.** It is one number per state,
measuring how uneven $|\psi|^2$ is, and it knows nothing about *which* orbit a state
sits on. The whole spread here is roughly 2.7 to 3.2 — real, reproducible, but not
dramatic. The sharp version projects $\psi$ onto a Gaussian tube around a specific
periodic orbit found by root-finding the bounce map, which is the "now push" step.

In [ ]:
h_scar = 0.01       # finer: scars are semiclassical, they need short wavelengths
n_scar = 300        # ~12 s at h_scar = 0.01; drop to 150 if that is too slow

scar_b = stadium(h=h_scar)
E_sc, V_sc = solve(scar_b, n_scar)
I_sc = ipr(V_sc, scar_b)
rank = np.argsort(I_sc)[::-1]          # most localised first

print(f"stadium at h={h_scar}: {n_scar} states on {scar_b.n_unknown} unknowns")
print(f"E up to {E_sc[-1]:.0f}, i.e. {2*np.pi/(np.sqrt(E_sc[-1])*h_scar):.1f} grid points "
      f"per wavelength at the top -- plenty to resolve the structure")
print(f"I*A: median {np.median(I_sc):.2f}   most localised {I_sc[rank[0]]:.2f} (k={rank[0]})")

BULK, HILITE, INK = "#8A94A6", "#C8553D", "#3A3F4B"
n_top, LAB_X = 12, 0.80          # LAB_X: where the reference labels start, axes fraction

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(E_sc, I_sc, s=14, color=BULK, alpha=0.75, linewidth=0)
ax.scatter(E_sc[rank[:n_top]], I_sc[rank[:n_top]], s=42, color=HILITE, zorder=3, linewidth=0)

# Label only the top few, and stagger the height: they sit at almost the same I*A,
# so a constant offset would overlap.
for i, k in enumerate(sorted(rank[:4], key=lambda k: E_sc[k])):
    ax.annotate(f"k={k}", (E_sc[k], I_sc[k]), textcoords="offset points",
                xytext=(0, 8 + 13 * (i % 2)), ha="center", fontsize=8, color=INK)

ax.set_xlim(0, E_sc[-1] / LAB_X)
ax.set_ylim(I_sc.min() - 0.05, I_sc.max() + 0.22)
for yref, lab in ((3.0, "GOE / Porter-Thomas"), (2.25, "rectangle mode, $9/4$")):
    ax.axhline(yref, xmax=LAB_X, color="0.6", ls="--", lw=0.9, zorder=0)
    ax.text(LAB_X + 0.02, yref, lab, transform=ax.get_yaxis_transform(),
            va="center", fontsize=8, color="0.4")

ax.set_xlabel("$E$")
ax.set_ylabel(r"$I \cdot A$   (localisation)")
ax.set_title(f"stadium: {n_top} most localised states highlighted", fontsize=11)
ax.grid(alpha=0.25, linewidth=0.6)
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
def plot_density(b, E, V, which, I=None, ncol=4, title=""):
    """|psi|^2 on a perceptually uniform sequential ramp.

    |psi|^2 is a MAGNITUDE, so it gets one ramp with monotonic lightness. A rainbow
    map (jet) has non-monotonic lightness and invents banding that is not in the data
    -- fatal here, where the question is literally 'is there structure'. The signed
    psi above gets a diverging map instead, because it has a meaningful zero.
    """
    nrow = int(np.ceil(len(which) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.4 * ncol, 2.1 * nrow), squeeze=False)
    flat = axes.ravel()
    for ax, k in zip(flat, which):
        rho = to_grid(V[:, k]**2, b.mask)
        ax.pcolormesh(b.x, b.y, rho.T, cmap="magma", shading="auto")
        tag = "" if I is None else f"   $I\\!\\cdot\\!A$={I[k]:.2f}"
        ax.set_title(f"k={k}   E={E[k]:.0f}{tag}", fontsize=8)
        ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    for ax in flat[len(which):]:
        ax.axis("off")
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.show()


plot_density(scar_b, E_sc, V_sc, rank[:12], I_sc,
             title="stadium: the 12 most localised states")
plot_density(scar_b, E_sc, V_sc, rank[-4:], I_sc,
             title="stadium: the least localised states -- ergodic, structureless speckle")

---

**Stop here.** Both shapes run the same solver. Still to come: a localisation measure
(inverse participation ratio) to rank states instead of flipping through pictures, and
a *quarter* stadium — the full stadium's two mirror symmetries split its spectrum into
four independent ones, which destroys the level statistics.